# 🌸 Tutorial Didático: Aprendizado Federado com Flower e PyTorch

Este notebook executa o treinamento federado no dataset **CIFAR-10** sincronizando diretamente com o repositório oficial do GitHub: **[JM3L0/flower_puro](https://github.com/JM3L0/flower_puro)**.

---
### ⚡ Configuração de Aceleração por GPU:
Certifique-se de que a GPU T4 do Colab está ativada:
No menu superior: **Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução -> Acelerador: T4 GPU -> Salvar**.

## 1. Verificar Disponibilidade da GPU T4

In [ ]:
!nvidia-smi

## 2. Puxar Código do Repositório GitHub e Instalar Dependências
Clona ou atualiza (`git pull`) o repositório **JM3L0/flower_puro** e instala as dependências oficiais.

In [ ]:
import os

REPO_URL = "https://github.com/JM3L0/flower_puro.git"
REPO_DIR = "/content/flower_puro"

# Se a pasta já existir na VM do Colab, atualiza via git pull; senão, clona do zero
if os.path.exists(REPO_DIR):
    print("🔄 Atualizando repositório para o commit mais recente (git pull)...")
    %cd {REPO_DIR}
    !git pull
else:
    print("📥 Clonando repositório do GitHub (JM3L0/flower_puro)...")
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

print("\n📦 Instalando dependências oficiais e pacote flower_puro...")
!pip install -q -r requirements.txt
!pip install -q -e .
print("\n✔ Ambiente pronto com o código mais recente do repositório!")

## 3. Painel de Controle Interativo e Execução da Simulação
Ajuste os parâmetros abaixo pelo formulário e execute o treinamento federado utilizando diretamente os módulos do repositório (`flower_puro` e `simulacao_direta.py`).

> 💡 **Dica de Execução:** Você pode usar o atalho **Ambiente de execução -> Executar tudo** (`Ctrl + F9`) para rodar todo o pipeline em sequência.

In [ ]:
#@title ⚙️ Painel de Controle da Simulação Federada { display-mode: "form" }
#@markdown ### 1. Estratégia de Agregação do Servidor
estrategia_agregacao = "FedAvg" #@param ["FedAvg", "FedMedian", "FedProx"]

#@markdown ---
#@markdown ### 2. Hiperparâmetros da Federação
num_rodadas = 10 #@param {type:"integer"}
num_clientes = 10 #@param {type:"integer"}
epocas_locais = 2 #@param {type:"integer"}
learning_rate = 0.01 #@param {type:"number"}
batch_size = 64 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Heterogeneidade dos Dados (Particionamento Dirichlet)
distribuicao_dados = "IID (Homogêneo - Alpha=100.0)" #@param ["IID (Homogêneo - Alpha=100.0)", "Não-IID Suave (Alpha=1.0)", "Não-IID Moderado (Alpha=0.5)", "Não-IID Severo (Alpha=0.1)"]

mapa_alpha = {
    "IID (Homogêneo - Alpha=100.0)": 100.0,
    "Não-IID Suave (Alpha=1.0)": 1.0,
    "Não-IID Moderado (Alpha=0.5)": 0.5,
    "Não-IID Severo (Alpha=0.1)": 0.1,
}
alpha_escolhido = mapa_alpha[distribuicao_dados]

# Garante ambiente e caminho corretos do repositório
import os, sys
REPO_DIR = "/content/flower_puro"
if os.path.exists(REPO_DIR):
    if os.getcwd() != REPO_DIR:
        os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    # Caso a Célula 2 tenha sido pulada pelo usuário
    print("⏳ Repositório não encontrado localmente. Clonando do GitHub...")
    !git clone https://github.com/JM3L0/flower_puro.git /content/flower_puro
    %cd /content/flower_puro
    !pip install -q -r requirements.txt
    !pip install -q -e .
    if "/content/flower_puro" not in sys.path:
        sys.path.insert(0, "/content/flower_puro")

# Executa a simulação utilizando diretamente os módulos do repositório (flower_puro)
from simulacao_direta import executar_simulacao

resumo = executar_simulacao(
    strategy=estrategia_agregacao,
    num_rounds=num_rodadas,
    num_clients=num_clientes,
    local_epochs=epocas_locais,
    learning_rate=learning_rate,
    batch_size=batch_size,
    dirichlet_alpha=alpha_escolhido,
    seed=seed,
    output_dir="resultados",
)

## 4. Visualização dos Gráficos e Tabela de Resultados
Exibe as curvas de aprendizado geradas (Acurácia Global e Loss) e a tabela de métricas consolidada.

In [ ]:
import os
from IPython.display import Image, Markdown, display

# 1. Exibe a figura de convergência gerada pelo script do repositório
caminho_grafico = "resultados/curvas_convergencia.png"
if os.path.exists(caminho_grafico):
    display(Image(filename=caminho_grafico, width=850))

# 2. Exibe o relatório em Markdown com as métricas por rodada
caminho_tabela = "resultados/TABELA_RESULTADOS.md"
if os.path.exists(caminho_tabela):
    with open(caminho_tabela, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

## 5. Download dos Resultados (Sem Dependência de Google Drive)
Baixa o arquivo compactado `.zip` contendo os pesos treinados (`modelo_federado.pt`), métricas JSON e o gráfico de convergência diretamente para a sua pasta Downloads.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/resultados_flower_puro", "zip", "resultados")
print("[+] Compactação concluída: /content/resultados_flower_puro.zip")
print("[*] Iniciando download pelo navegador...")
files.download("/content/resultados_flower_puro.zip")